# The SQLite Search Cache
### Harness Engineering · Course 4: Search Engineering — Advanced

**Learning objective:** Implement a `cached_search()` wrapper backed by SQLite that normalises query strings to cache keys via SHA-256, stores results with configurable TTL, serves cache hits without calling the API, and evicts expired entries on every write.

---

**Sections**
1. Setup & install
2. The broken baseline — 12 API calls, 4 novel
3. The fix — full `search_cache` implementation
4. End-to-end verification — before vs. after
5. Exercises

## 1. Setup

In [ ]:
%pip install -q duckduckgo-search

In [ ]:
import hashlib
import json
import os
import sqlite3
import time
from collections.abc import Callable
from duckduckgo_search import DDGS

print("Ready.")

---
## 2. The Broken Baseline

This stub simulates three autoresearch sessions, each issuing the same four queries.

**Before reading further:** Run the cell, count the API calls, and answer —  
*how many of those calls carried new information?*

In [ ]:
api_calls = 0

def search(query: str, max_results: int = 5) -> list[dict]:
    """Raw search — no caching. Every call hits the API."""
    global api_calls
    api_calls += 1
    print(f"[API CALL #{api_calls:02d}] {query[:60]}")
    return list(DDGS().text(query, max_results=max_results))

QUERIES = [
    "transformer attention mechanisms survey",
    "KV cache optimization techniques",
    "flash attention implementation pytorch",
    "multi-head latent attention MLA",
]

api_calls = 0
for session in range(1, 4):
    print(f"\n=== Session {session} ===")
    for q in QUERIES:
        search(q)
    time.sleep(1.5)   # avoid rate limits between sessions

print(f"\nTotal API calls : {api_calls}")
print(f"Unique queries  : {len(QUERIES)}")
print(f"Redundant calls : {api_calls - len(QUERIES)}")

**Expected output:** 12 total API calls, 4 unique. Sessions 2 and 3 paid API cost and wall time to retrieve results already available from session 1. No error is raised. The waste is silent.

---
## 3. The Fix — `search_cache` Implementation

### 3.1 Query normalisation

SHA-256 is case-sensitive and whitespace-sensitive. `"Flash Attention"` and `"flash attention"` produce completely different 256-bit digests. Normalise *before* hashing:

In [ ]:
# Demonstration — same intent, different raw strings -> different digests without normalisation
raw_a = "Flash Attention"
raw_b = "flash attention"
raw_c = "flash  attention"  # double space

for q in [raw_a, raw_b, raw_c]:
    digest = hashlib.sha256(q.encode()).hexdigest()[:16]
    print(f"{q!r:25s}  ->  {digest}...")

print()
print("After normalisation:")
for q in [raw_a, raw_b, raw_c]:
    normalised = " ".join(q.lower().split())
    digest = hashlib.sha256(normalised.encode()).hexdigest()[:16]
    print(f"{q!r:25s}  ->  {digest}...  (normalised: {normalised!r})")

### 3.2 Full implementation

In [ ]:
DB_PATH     = "/tmp/search_cache.db"   # Colab-safe temp path
DEFAULT_TTL = 86_400                   # 24 hours


def _connect() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS search_cache (
            key        TEXT PRIMARY KEY,
            query      TEXT NOT NULL,
            results    TEXT NOT NULL,
            created_at REAL NOT NULL DEFAULT 0,
            expires_at REAL NOT NULL DEFAULT 0
        )
    """)
    conn.execute(
        "CREATE INDEX IF NOT EXISTS idx_expires ON search_cache(expires_at)"
    )
    conn.commit()
    return conn


def _cache_key(query: str) -> str:
    """Normalise then hash. Same intent always maps to the same key."""
    normalised = " ".join(query.lower().split())
    return hashlib.sha256(normalised.encode()).hexdigest()


def get(query: str) -> list[dict] | None:
    """Return cached results, or None on MISS or expiry."""
    key  = _cache_key(query)
    now  = time.time()
    conn = _connect()
    try:
        row = conn.execute(
            "SELECT results, expires_at FROM search_cache WHERE key = ?",
            (key,)
        ).fetchone()
        if row is None:
            return None
        results_json, expires_at = row
        if expires_at < now:
            conn.execute("DELETE FROM search_cache WHERE key = ?", (key,))
            conn.commit()
            return None
        return json.loads(results_json)
    finally:
        conn.close()


def put(query: str, results: list[dict], ttl: int = DEFAULT_TTL) -> None:
    """Upsert results. Inline-evict stale rows on every write."""
    key  = _cache_key(query)
    now  = time.time()
    conn = _connect()
    try:
        conn.execute(
            """
            INSERT INTO search_cache (key, query, results, created_at, expires_at)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(key) DO UPDATE SET
                results    = excluded.results,
                created_at = excluded.created_at,
                expires_at = excluded.expires_at
            """,
            (key, query, json.dumps(results, ensure_ascii=False), now, now + ttl),
        )
        # Inline eviction — keeps DB bounded without a background timer
        conn.execute("DELETE FROM search_cache WHERE expires_at < ?", (now,))
        conn.commit()
    finally:
        conn.close()


def cached_search(
    query: str,
    search_fn: Callable[[str, int], list[dict]],
    ttl: int = DEFAULT_TTL,
    max_results: int = 5,
) -> list[dict]:
    """Drop-in wrapper: one-line callsite change from search(q) -> cached_search(q, search)."""
    cached = get(query)
    if cached is not None:
        print(f"[cache HIT ] {query[:60]}")
        return cached
    print(f"[cache MISS] {query[:60]}")
    results = search_fn(query, max_results)
    if results:
        put(query, results, ttl=ttl)
    return results


def stats() -> dict:
    """Inspect cache state."""
    now  = time.time()
    conn = _connect()
    try:
        total   = conn.execute("SELECT COUNT(*) FROM search_cache").fetchone()[0]
        expired = conn.execute(
            "SELECT COUNT(*) FROM search_cache WHERE expires_at < ?", (now,)
        ).fetchone()[0]
        size_b  = os.path.getsize(DB_PATH) if os.path.exists(DB_PATH) else 0
        return {"total": total, "live": total - expired,
                "expired": expired, "size_kb": size_b // 1024}
    finally:
        conn.close()


print("search_cache module defined.")

---
## 4. End-to-End Verification

Replace `search()` with `cached_search()`. Re-run 3 sessions.  
**Expected:** Session 1 → 4 MISS (4 API calls). Sessions 2 & 3 → 4 HIT each (0 API calls).

In [ ]:
# Reset the DB so the demo starts cold
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

api_calls = 0

def search_with_counter(query: str, max_results: int = 5) -> list[dict]:
    global api_calls
    api_calls += 1
    return list(DDGS().text(query, max_results=max_results))

for session in range(1, 4):
    print(f"\n=== Session {session} ===")
    for q in QUERIES:
        cached_search(q, search_with_counter)
    time.sleep(0.5)

print(f"\nTotal API calls : {api_calls}   (was 12 without cache)")
print(f"Unique queries  : {len(QUERIES)}")
print(f"Calls saved     : {12 - api_calls}")
print()
print("Cache state:", stats())

### TTL Behaviour

In [ ]:
# Demonstrate TTL expiry with a 3-second TTL
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

SHORT_TTL = 3  # seconds

print("First call (MISS -> stored with 3s TTL):")
cached_search(QUERIES[0], lambda q, n: list(DDGS().text(q, max_results=3)), ttl=SHORT_TTL)

print("\nImmediate second call (HIT):")
cached_search(QUERIES[0], lambda q, n: list(DDGS().text(q, max_results=3)), ttl=SHORT_TTL)

print("\nWaiting 4 seconds for TTL to expire...")
time.sleep(4)

print("Third call after expiry (MISS again):")
cached_search(QUERIES[0], lambda q, n: list(DDGS().text(q, max_results=3)), ttl=SHORT_TTL)

---
## 5. Exercises

### Exercise A — Implement from the starter skeleton

Implement the three functions below from scratch using only the docstrings as a guide.

In [ ]:
def _cache_key_ex(query: str) -> str:
    # TODO: lowercase, collapse whitespace, return SHA-256 hex digest
    pass


def get_ex(query: str) -> list[dict] | None:
    # TODO: lookup key, check expiry, return json.loads(results_json) or None
    pass


def put_ex(query: str, results: list[dict], ttl: int = DEFAULT_TTL) -> None:
    # TODO: upsert with ON CONFLICT(key) DO UPDATE
    # TODO: inline eviction: DELETE WHERE expires_at < now
    pass


# Quick self-test (uncomment after implementing)
# assert _cache_key_ex("Flash Attention") == _cache_key_ex("flash attention")
# assert _cache_key_ex("flash  attention") == _cache_key_ex("flash attention")
# print("_cache_key_ex: PASS")

### Exercise B — Scale calculation

Your autoresearch loop runs 12 queries per session across 10 sessions per topic per day, on 8 active research topics.

1. How many API calls does the uncached version make?
2. After adding the cache (assume all 12 queries repeat across sessions), how many API calls does session 2+ make per topic?
3. What is the daily API call reduction across all topics?

In [ ]:
queries_per_session = 12
sessions_per_topic  = 10
topics              = 8

# TODO: fill in the calculations
uncached_total = None
cached_total   = None
calls_saved    = None

print(f"Uncached total : {uncached_total}")
print(f"Cached total   : {cached_total}")
print(f"Calls saved    : {calls_saved}")

### Exercise C — Production hardening (optional)

Extend the implementation with a `clear_expired()` helper and a `stats()` dashboard that reports total rows, live rows, expired rows, and database size. Integrate into the session loop so stats are printed after each session.

In [ ]:
def clear_expired() -> int:
    """Delete all expired rows. Returns the number of rows removed."""
    # TODO
    pass